# TabPFN drift - ADWIN sensitivity ablation (delta)
ADWIN's **delta** is its significance level: a smaller delta needs more evidence before it flags
drift, so it flags less often. The main results use delta = 0.002 (already run, in `results/`).
Here we run **real TabPFN** with the ADWIN policy (P3) only, for delta = **1e-4, 1e-6, 1e-10** on:
- synthetic **sudden** and **gradual** drift, seeds 0, 1, 2
- real **Bike Sharing** and **Appliances Energy**, seed 0 only (so no confidence interval there)

Outputs go to `/content/results_ablation` (no Google Drive), with names like
`sudden_0_P3_drift_d1e-4.csv`. **After every dataset, run its download cell** and keep the zip:
if Colab disconnects, `/content` is wiped. To continue: run the setup cells again, then the
*restore* cell with your latest zip; finished CSVs are skipped.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L

## 2. Upload the code zip (includes `data/raw` for the real datasets and `results/` for the default-delta runs)
Make it on your laptop in PowerShell, inside the repo folder (skips `.venv`):
```
tar.exe -a -cf ..\tabpfn-drift.zip --exclude=__pycache__ common data eval methods results notebooks run_experiments.py smoke_test.py time_tabpfn.py plot_check.py requirements.txt
```
Then pick `tabpfn-drift.zip` (one folder up from the repo) when the upload box appears.

In [ ]:
from google.colab import files
files.upload()
!rm -rf /content/tabpfn-drift && mkdir -p /content/tabpfn-drift
!unzip -o -q tabpfn-drift.zip -d /content/tabpfn-drift
%cd /content/tabpfn-drift
!ls data/raw
!grep -c "adwin-delta" run_experiments.py   # must print a number > 0 (the zip has the ablation option)

## 3. Install packages (numpy, pandas, sklearn, torch are already on Colab)

In [ ]:
!pip install -q tabpfn river

## 4. TabPFN token (never paste it into a code cell or commit it)
Route A: licence key from https://ux.priorlabs.ai -> `TABPFN_TOKEN`.
Route B: accept the TabPFN model terms on huggingface.co, make a *Read* token -> `HF_TOKEN`.
Press Enter to skip either one.

In [ ]:
import os, getpass
t = getpass.getpass('TABPFN_TOKEN (Enter to skip): ').strip()
if t: os.environ['TABPFN_TOKEN'] = t
h = getpass.getpass('HF_TOKEN (Enter to skip): ').strip()
if h: os.environ['HF_TOKEN'] = h

## 5. Settings + download helper (used after every dataset)

In [ ]:
import os, shutil
from google.colab import files
OUT = '/content/results_ablation'
os.makedirs(OUT, exist_ok=True)
CMAX = 2000
DELTAS = ['1e-4', '1e-6', '1e-10']      # 0.002 is the default, already in results/

def download_results(stage):
    """Zip everything in OUT and download it to your laptop."""
    n = len([f for f in os.listdir(OUT) if f.endswith('.csv')])
    path = shutil.make_archive(f'/content/results_ablation_{stage}', 'zip', '/content', 'results_ablation')
    print(f'{n} result CSVs -> {path}')
    files.download(path)

def run(where, seeds):
    """ADWIN only, one call per delta. Resumable: finished CSVs are skipped."""
    for d in DELTAS:
        get_ipython().system(f'python run_experiments.py {where} --seeds {seeds} --methods adwin '
                             f'--adwin-delta {d} --c-max {CMAX} --out {OUT}')

## 6. (Only after a disconnect) restore your last downloaded `results_ablation_*.zip`

In [ ]:
for name in files.upload():
    shutil.unpack_archive(name, '/content')
print(sorted(os.listdir(OUT)))

## 7. Pipeline self-test (stand-in model, not TabPFN). Must end with ALL SMOKE TESTS PASSED

In [ ]:
%cd /content/tabpfn-drift
!python smoke_test.py 2>&1 | tail -8

## 8. Time TabPFN (also proves the token works). Note the seconds for 2000 rows.

In [ ]:
!python time_tabpfn.py

## 9. Rough time estimate
Based on the synthetic runs: vanilla took ~115 s per 59 blocks. ADWIN costs at most about the same
per block (it predicts once per block; after a drift flag its context is smaller, so it is faster).
If cell 8 shows a different time for 2000 rows, put it in `PER_BLOCK_S` and re-run this cell.

In [ ]:
from data.real import load_real
PER_BLOCK_S = 115 / 59                      # seconds per block (upper bound for ADWIN)
blocks = {'sudden': 59, 'gradual': 59,
          'bike': load_real('bike').df.block.nunique() - 1,
          'energy': load_real('energy').df.block.nunique() - 1}
seeds = {'sudden': 3, 'gradual': 3, 'bike': 1, 'energy': 1}
total = 0
for ds in blocks:
    m = blocks[ds] * seeds[ds] * len(DELTAS) * PER_BLOCK_S / 60
    total += m
    print(f"{ds:7s}: {seeds[ds]} seed(s) x {len(DELTAS)} deltas x {blocks[ds]} blocks -> ~{m:.0f} min")
print(f"total: ~{total:.0f} min")

## Dataset 1 - synthetic sudden drift (drift at block 30), seeds 0-2

In [ ]:
run('--streams sudden', '0 1 2')

In [ ]:
download_results('1_sudden')

## Dataset 2 - synthetic gradual drift (blocks 20 to 40), seeds 0-2

In [ ]:
run('--streams gradual', '0 1 2')

In [ ]:
download_results('2_gradual')

## Dataset 3 - Bike Sharing (real), seed 0

In [ ]:
run('--dataset bike', '0')

In [ ]:
download_results('3_bike')

## Dataset 4 - Appliances Energy (real), seed 0

In [ ]:
run('--dataset energy', '0')

In [ ]:
download_results('4_energy')

## Ablation table + chart
Compares the new deltas with the default-delta ADWIN and vanilla runs from `results/` in the zip.

In [ ]:
%cd /content/tabpfn-drift
!python -m eval.make_ablation --ablation {OUT} --main results
from IPython.display import Image
Image(f'{OUT}/figures/adwin_delta.png')

In [ ]:
download_results('5_final')

**When you are done:** unzip the last zip into `results_ablation/` in the repo (not into `results/`),
then run `python -m eval.make_ablation` on your laptop.